# 05 - Creation of negative class

In [71]:
ENV = "local"

from pathlib import Path

if ENV == "colab":
    DATA_DIR = Path("/content/drive/MyDrive/thesis")
else:
    DATA_DIR = Path("00_data")

if ENV == "colab":
    from google.colab import drive
    drive.mount('/content/drive')


In [72]:
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split

In [73]:
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from general.util.banned_words import add_flagged_column_vectorized

# Data

In [74]:
dgf_dedup = pd.read_parquet(str(DATA_DIR / '04_dgf_dedup_nomic.parquet'))
dgf_dedup

,title,description,publisher,identifier,slug,has_spatial,popularity,last_harvested_date,keyword,theme,...,dcat_program_code,dcat_periodicity,dcat_license,dcat_rights,title_norm,desc_len,desc_norm,title_topic,desc_embeddings_nomic,title_embeddings_nomic
0,2013-005_299BPHOTOGRAPHS: SEABOSS Images from ...,"The U.S. Geological Survey (USGS), in cooperat...",U.S. Geological Survey,http://datainventory.doi.gov/id/dataset/USGS_1...,2013-005_299bphotographs-seaboss-images-from-t...,True,0,2026-01-27T03:12:54.564398,"[Atlantic Ocean, Block Island Sound, CMGP, Coa...",[Geospatial],...,[],,,,2013 005 299bphotographs seaboss images from t...,853,the us geological survey usgs cooperation with...,NaN,"[0.0391845703125, 0.0283966064453125, -0.19165...","[-0.0239410400390625, 0.036712646484375, -0.19..."
1,"006 - Santa Cruz Harbor, CA","Timeseries data from '006 - Santa Cruz Harbor,...",Axiom Data Science,edu_ucsd_cdip_006,006-santa-cruz-harbor-ca,True,2,2026-01-26T23:14:29.853735,"[earth science, atmosphere, ocean, biosphere, ...",[],...,[],,https://creativecommons.org/publicdomain/zero/...,,006 santa cruz harbor ca,70,timeseries data from 006 santa cruz harbor ca ...,NaN,"[-0.01556396484375, 0.0185394287109375, -0.239...","[-0.03631591796875, 0.0325927734375, -0.228637..."
2,01) Revised groundwater-level contours for Smi...,These contours represent static groundwater le...,U.S. Geological Survey,http://datainventory.doi.gov/id/dataset/USGS_6...,01-revised-groundwater-level-contours-for-smit...,True,1,2026-01-27T03:18:03.546706,"[Great Basin, Groundwater, Lyon County, Mason ...",[Geospatial],...,[],,,,01 revised groundwater level contours for smit...,1352,these contours represent static groundwater le...,NaN,"[0.0408935546875, 0.0119781494140625, -0.22399...","[0.020660400390625, -0.048126220703125, -0.197..."
3,01. Seismicity catalogs for the 2023 update of...,This dataset contains earthquake catalogs comp...,U.S. Geological Survey,http://datainventory.doi.gov/id/dataset/USGS_6...,01-seismicity-catalogs-for-the-2023-update-of-...,True,3,2026-01-27T03:57:37.770642,"[Alaska, NSHM, NSHMP, National Seismic Hazard ...",[Geospatial],...,[],,,,01 seismicity catalogs for the 2023 update of ...,536,this dataset contains earthquake catalogs comp...,NaN,"[0.036376953125, 0.040985107421875, -0.2089843...","[-0.04022216796875, -0.010894775390625, -0.197..."
4,01: Watersheds shapefile for the 15 study wate...,This Geographic Information System dataset con...,U.S. Geological Survey,http://datainventory.doi.gov/id/dataset/USGS_6...,01-watersheds-shapefile-for-the-15-study-water...,True,0,2026-01-27T02:17:38.778671,"[Gwinnett County, State of Georgia, USGS:63502...",[Geospatial],...,[],,,,01 watersheds shapefile for the 15 study water...,234,this geographic information system dataset con...,NaN,"[0.01325225830078125, 0.0146026611328125, -0.1...","[-0.0289459228515625, -0.0162353515625, -0.173..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269078,Wyoming (WY),This report presents Wyoming\'s estimates for ...,Substance Abuse and Mental Health Services Adm...,https://healthdata.gov/api/views/7756-egub,wyoming-wy-1b2b8,False,0,2025-09-07T09:21:31.075371,"[mental-health, samhsa, substance-use, suicida...",[SAMHSA],...,[009:061],,,,wyoming wy,600,this report presents wyomings estimates for 25...,,"[0.0084991455078125, -0.005733489990234375, -0...","[-0.0229034423828125, 0.004611968994140625, -0..."
269079,Missouri (MO),This report uses 2008 to 2010 National Survey ...,Substance Abuse and Mental Health Services Adm...,https://healthdata.gov/api/views/36za-n73h,missouri-mo,False,0,2025-09-07T07:24:52.464410,"[mental-health-indicators, missouri, nsduh-dat...",[SAMHSA],...,[009:061],,,,missouri mo,536,this report uses to national survey on drug us...,,"[0.0137786865234375, -0.0355224609375, -0.1898...","[0.00608062744140625, -0.036407470703125, -0.1..."
269080,Maine (ME),This report presents Maine\'s estimates for 25...,S

In [75]:
drp = pd.read_parquet(str(DATA_DIR / '04_drp_in_dgf_nomic.parquet'))

# Make negative class

identifying DRP entries

In [76]:
drp_titles = set(drp['title'].dropna())
drp_identifiers = set(drp['identifier'].dropna())
drp_slugs = set(drp['slug'].dropna())
print(f'DRP entries to exclude: {len(drp):,}')
print(f' titles:  {len(drp_titles):,}')
print(f' identifiers: {len(drp_identifiers):,}')
print(f' slugs: {len(drp_slugs):,}')


DRP entries to exclude: 1,313
 titles:  1,192
 identifiers: 1,313
 slugs: 1,313


In [77]:
def id_drp(df):
    return (
        df['title'].isin(drp_titles)
        | df['identifier'].isin(drp_identifiers)
        | df['slug'].isin(drp_slugs)
    )

* federal_all - organisation type is federal
* non-federal - everythign other than federal_all
* non_drp - no drp identifiers

In [78]:
federal_all = dgf_dedup[dgf_dedup['org_type'].str.lower().str.contains('federal', na=False)].copy()
non_federal_all = dgf_dedup[~dgf_dedup['org_type'].str.lower().str.contains('federal', na=False)].copy()
non_drp = dgf_dedup.copy()

federal_all = federal_all[~id_drp(federal_all)]
non_federal_all = non_federal_all[~id_drp(non_federal_all)]
non_drp = non_drp[~id_drp(non_drp)]

print(f'federal_all: {len(federal_all):,}')
print(f'non_federal_all: {len(non_federal_all):,}')
print(f'non_drp: {len(non_drp):,}')

federal_all: 247,478
non_federal_all: 20,622
non_drp: 268,100


In [79]:
non_drp.to_parquet(str(DATA_DIR / '05_nondrp.parquet'))

`non_drp` df is exported to be used in the PU learning notebook (08_pulearning)

### CLASS 1
DRP positives

In [10]:
class1_df = drp.copy()
class1_df['label'] = 1

### CLASS 0

#### Broad pool
Org type distribution matched to DRP -  99% federal, 1% non-federal

In [11]:
drp['org_type'].value_counts(normalize=True)

org_type
Federal Government    0.989337
State Government      0.006093
City Government       0.003808
County Government     0.000762
Name: proportion, dtype: float64

In [15]:
n_class0 = len(class1_df)
n_fed = int(n_class0 * 0.99)
n_nonfed = n_class0 - n_fed

c0_fed = federal_all.sample(n=min(n_fed, len(federal_all)), random_state=42)
c0_nonfed = non_federal_all.sample(n=min(n_nonfed, len(non_federal_all)), random_state=42)

class0_broad_pool = pd.concat([c0_fed, c0_nonfed], ignore_index=True)
class0_broad_pool['label'] = 0

print(f'Total: {len(class0_broad_pool):,}  (federal: {len(c0_fed):,}, non-federal: {len(c0_nonfed):,})')

Total: 1,313  (federal: 1,299, non-federal: 14)


#### No banned words
same ratio but only entries with 0 flagged words in title or description

In [17]:
fed_flagged = add_flagged_column_vectorized(federal_all, abstract_col='description', title_col='title')
nonfed_flagged = add_flagged_column_vectorized(non_federal_all, abstract_col='description', title_col='title')

federal_no_banned = fed_flagged[fed_flagged['has_flagged_word_total'] == 0]
non_federal_no_banned = nonfed_flagged[nonfed_flagged['has_flagged_word_total'] == 0]

In [18]:
federal_no_banned.to_parquet(str(DATA_DIR / '05_c0_fed_nb.parquet'))
non_federal_no_banned.to_parquet(str(DATA_DIR / '05_c0_nonfed_nb.parquet'))

In [19]:
federal_no_banned = pd.read_parquet(str(DATA_DIR / '05_c0_fed_nb.parquet'))
non_federal_no_banned = pd.read_parquet(str(DATA_DIR / '05_c0_fed_nb.parquet'))

makign the class 0 pool no banned

In [20]:
c0_fed_nb = federal_no_banned.sample(n=min(n_fed, len(federal_no_banned)), random_state=42)
c0_nonfed_nb = non_federal_no_banned.sample(n=min(n_nonfed, len(non_federal_no_banned)), random_state=42)

class0_broad_pool_no_banned = pd.concat([c0_fed_nb, c0_nonfed_nb], ignore_index=True)
class0_broad_pool_no_banned['label'] = 0

In [21]:
print(f'federal_no_banned:     {len(federal_no_banned):,} / {len(federal_all):,}'
      f'  ({len(federal_no_banned)/len(federal_all)*100:.1f}% retained)')
print(f'non_federal_no_banned: {len(non_federal_no_banned):,} / {len(non_federal_all):,}'
      f'  ({len(non_federal_no_banned)/len(non_federal_all)*100:.1f}% retained)')
print(f'Class 0 broad pool (no banned): {len(class0_broad_pool_no_banned):,}'
      f'  (federal: {len(c0_fed_nb):,}, non-federal: {len(c0_nonfed_nb):,})')

federal_no_banned:     187,409 / 247,478  (75.7% retained)
non_federal_no_banned: 187,409 / 20,622  (908.8% retained)
Class 0 broad pool (no banned): 1,313  (federal: 1,299, non-federal: 14)


#### Same agency (with banned)
distribution matched

* part 1: matching drp publisher distribution

* part 2: adding more entries proportionally to representation in drp if there is surplus

In [23]:
drp_pub_counts = drp['publisher'].value_counts()
n_pos = len(drp)

phase1_parts = []
surplus = {}

for pub, count in drp_pub_counts.items():
    pool = non_drp[non_drp['publisher'] == pub]
    if len(pool) == 0:
        continue
    take = min(count, len(pool))
    sampled = pool.sample(n=take, random_state=42)
    phase1_parts.append(sampled)
    remaining = pool.drop(sampled.index)
    if len(remaining) > 0:
        surplus[pub] = remaining

phase1 = pd.concat(phase1_parts, ignore_index=True)
n_needed = n_pos - len(phase1)
print(f"distribution-matched: {len(phase1):,},  need {n_needed:,} more")

distribution-matched: 699,  need 614 more


In [24]:
eligible_drp_counts = drp_pub_counts[list(surplus.keys())]
weights = eligible_drp_counts / eligible_drp_counts.sum()
raw_alloc = (weights * n_needed).round().astype(int)
capped_alloc = {pub: min(raw_alloc[pub], len(surplus[pub])) for pub in surplus}

print("part 2 allocation - top publishers by DRP count:")
for pub in sorted(surplus, key=lambda p: -drp_pub_counts[p])[:15]:
    print(f"  {pub[:55]:<55} drp={drp_pub_counts[pub]:4d}  "
          f"surplus={len(surplus[pub]):6d}  "
          f"target={raw_alloc[pub]:4d}  take={capped_alloc[pub]:4d}")

phase2_parts = [surplus[pub].sample(n=n, random_state=42)
                for pub, n in capped_alloc.items() if n > 0]
phase2 = pd.concat(phase2_parts, ignore_index=True) if phase2_parts else pd.DataFrame()

total = len(phase1) + len(phase2)
print(f"part 2 proportional top up: {len(phase2):,}")
if total < n_pos:
    print(f"surplus exhausted — pool is {total:,} (short by {n_pos - total:,})")

part 2 allocation - top publishers by DRP count:
  HIFLD                                                   drp= 145  surplus=    67  target= 159  take=  67
  Department of Veterans Affairs                          drp= 103  surplus=   930  target= 113  take= 113
  U.S. Geological Survey                                  drp=  97  surplus= 28174  target= 106  take= 106
  Centers for Medicare & Medicaid Services                drp=  48  surplus=   473  target=  53  take=  53
  U.S. Department of Housing and Urban Development        drp=  19  surplus=   149  target=  21  take=  21
  Bureau of Transportation Statistics                     drp=  18  surplus=   208  target=  20  take=  20
  SEDAC                                                   drp=  15  surplus=   238  target=  16  take=  16
  Bureau of Transportation Statistics (BTS)               drp=   9  surplus=    18  target=  10  take=  10
  Federal Housing Finance Agency                          drp=   8  surplus=    23  target=   9

In [25]:
class0_sa_pool = pd.concat([phase1, phase2], ignore_index=True)
class0_sa_pool['label'] = 0

print(f"\nClass 1 (DRP positives): {n_pos:,}")
print(f"Class 0 SA pool (total): {len(class0_sa_pool):,}")
print(f"Ratio: {len(class0_sa_pool)/n_pos:.2f}x")


Class 1 (DRP positives): 1,313
Class 0 SA pool (total): 1,201
Ratio: 0.91x


# Broad pool (no banned)

In [48]:
def col(col, fill=''):
    return df[col].fillna(fill) if col in df.columns else pd.Series(fill, index=df.index)

def list_col(col):
    return df[col] if col in df.columns else pd.Series([[] for _ in range(len(df))], index=df.index)

def has_content(x):
    return int(hasattr(x, '__len__') and not isinstance(x, str) and len(x) > 0)

In [49]:
df = pd.concat([class1_df, class0_broad_pool_no_banned], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

text properties

In [50]:
df['desc_length'] = col('description').str.len()
df['title_length'] = col('title').str.len()
df['desc_word_count'] = col('description').str.split().str.len()
df['all_text'] = col('title') + ' ' + col('description')

engagement

In [51]:
df['has_spatial'] = df['has_spatial'].fillna(False).astype(int) if 'has_spatial' in df.columns else 0
df['popularity'] = pd.to_numeric(col('popularity', 0), errors='coerce').fillna(0)
df['n_keywords'] = list_col('keyword').apply(lambda x: len(x) if hasattr(x, '__len__') and not isinstance(x, str) else 0)
df['n_themes'] = list_col('theme').apply(lambda x: len(x) if hasattr(x, '__len__') and not isinstance(x, str) else 0)

temporal

In [52]:
df['has_issued'] = (col('dcat_issued').str.strip()   != '').astype(int)
df['has_modified'] = (col('dcat_modified').str.strip() != '').astype(int)

completeness

In [53]:
df['has_license'] = (col('dcat_license').str.strip()   != '').astype(int)
df['has_bureau_code'] = list_col('dcat_bureau_code').apply(has_content)
df['has_program_code'] = list_col('dcat_program_code').apply(has_content)
df['has_temporal_info'] = (col('dcat_temporal').str.strip()  != '').astype(int)
df['has_periodicity'] = (col('dcat_periodicity').str.strip() != '').astype(int)
df['has_landing_page'] = (col('dcat_landing_page').str.strip() != '').astype(int)

org

In [54]:
df['org_type_federal'] = col('org_type').str.lower().str.contains('federal', na=False).astype(int)
df['dcat_access_level'] = col('dcat_access_level', 'unknown')

In [ ]:
# df = df.drop(columns=['desc_len', 'desc_norm'])

In [55]:
df = add_flagged_column_vectorized(df, abstract_col='description', title_col='title')

In [56]:
df['flagged_ratio'] = (df['total_flagged_terms_total'] / df['desc_word_count'].replace(0, np.nan)).fillna(0)

In [57]:
df.to_parquet(str(DATA_DIR / '05_broad_nb.parquet'))

In [58]:
df = pd.read_parquet(str(DATA_DIR / '05_broad_nb.parquet'))

# Same agency
using class 0 same agency pool instead of class 0 broad pool

In [59]:
def col_sa(col, fill=''):
    return df_sa[col].fillna(fill) if col in df_sa.columns else pd.Series(fill, index=df_sa.index)

def list_col_sa(col):
    return df_sa[col] if col in df_sa.columns else pd.Series([[] for _ in range(len(df_sa))], index=df_sa.index)

def has_content_sa(x):
    return int(hasattr(x, '__len__') and not isinstance(x, str) and len(x) > 0)

In [60]:
df_sa = pd.concat([class1_df, class0_sa_pool], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

text properties

In [61]:
df_sa['desc_length'] = col_sa('description').str.len()
df_sa['title_length'] = col_sa('title').str.len()
df_sa['desc_word_count']= col_sa('description').str.split().str.len()
df_sa['all_text'] = col_sa('title') + ' ' + col_sa('description')

engagement

In [62]:
df_sa['has_spatial'] = df_sa['has_spatial'].fillna(False).astype(int) if 'has_spatial' in df_sa.columns else 0
df_sa['popularity'] = pd.to_numeric(col_sa('popularity', 0), errors='coerce').fillna(0)
df_sa['n_keywords'] = list_col_sa('keyword').apply(lambda x: len(x) if hasattr(x, '__len__') and not isinstance(x, str) else 0)
df_sa['n_themes']  = list_col_sa('theme').apply(lambda x: len(x) if hasattr(x, '__len__') and not isinstance(x, str) else 0)

temporal

In [63]:
df_sa['has_issued'] = (col_sa('dcat_issued').str.strip()   != '').astype(int)
df_sa['has_modified'] = (col_sa('dcat_modified').str.strip() != '').astype(int)

completeness

In [64]:
df_sa['has_license'] = (col_sa('dcat_license').str.strip()      != '').astype(int)
df_sa['has_bureau_code'] = list_col_sa('dcat_bureau_code').apply(has_content_sa)
df_sa['has_program_code'] = list_col_sa('dcat_program_code').apply(has_content_sa)
df_sa['has_temporal_info']= (col_sa('dcat_temporal').str.strip()  != '').astype(int)
df_sa['has_periodicity'] = (col_sa('dcat_periodicity').str.strip() != '').astype(int)
df_sa['has_landing_page'] = (col_sa('dcat_landing_page').str.strip()!= '').astype(int)

org

In [65]:
df_sa['org_type_federal'] = col_sa('org_type').str.lower().str.contains('federal', na=False).astype(int)
df_sa['dcat_access_level'] = col_sa('dcat_access_level', 'unknown')

In [66]:
df_sa = add_flagged_column_vectorized(df_sa, abstract_col='description', title_col='title')
df_sa['flagged_ratio'] = (df_sa['total_flagged_terms_total'] / df_sa['desc_word_count'].replace(0, np.nan)).fillna(0)

In [67]:
df_sa.shape

(2514, 70)

In [68]:
df_sa.to_parquet(str( DATA_DIR / '05_sa.parquet'))

In [69]:
df_sa = pd.read_parquet(str( DATA_DIR / '05_sa.parquet'))
df_sa

,title,description,publisher,identifier,slug,has_spatial,popularity,last_harvested_date,keyword,theme,...,unique_pen_words_title,unique_nyt_words_title,flagged_words_all_abstract,flagged_words_all_title,total_flagged_terms_total,unique_flagged_words_total,has_flagged_word_total,has_flagged_terms_total,avg_repetition,flagged_ratio
0,NNDSS - TABLE 1M. Gonorrhea to Haemophilus inf...,NNDSS - TABLE 1M. Gonorrhea to Haemophilus inf...,Centers for Disease Control and Prevention,https://data.cdc.gov/api/views/h4wb-nae4,nndss-table-1m-gonorrhea-to-haemophilus-influe...,0,0,2025-08-08T15:20:14.743931,"[2019, age <5 years, all ages, all serotypes, ...",[NNDSS],...,0,0,,,0,0,0,0,0.0,0.000000
1,Environmental Protection Agency (EPA) Regions,Homeland Infrastructure Foundation-Level Data ...,HIFLD,MGMT-GMO-HIFLD-677626,environmental-protection-agency-epa-regions,0,0,2025-08-02T09:16:41.425325,[none],[],...,0,0,,,0,0,0,0,0.0,0.000000
2,NNDSS - Table II. West Nile virus disease,NNDSS - Table II. West Nile virus disease - 20...,Centers for Disease Control and Prevention,https://data.cdc.gov/api/views/sd5c-m3g5,nndss-table-ii-west-nile-virus-disease-e9366,0,0,2025-08-09T04:32:37.333085,"[2016, mmwr, nedss, netss, nndss, west nile vi...",[NNDSS],...,0,0,,,0,0,0,0,0.0,0.000000
3,COVID-19 Vaccination Trends in the United Stat...,Overall Trends in Number of COVID-19 Vaccinati...,Centers for Disease Control and Prevention,https://data.cdc.gov/api/views/rh2h-3yt2,covid-19-vaccination-trends-in-the-united-stat...,1,0,2025-08-09T03:35:34.294662,"[administration, coronavirus, covid-19, immuni...",[Vaccinations],...,1,0,covid-19,covid-19,0,2,1,0,0.0,0.000000
4,American Time Use Survey,The American Time Use Survey (ATUS) provides n...,Bureau of Labor Statistics,DOL-BLS-218,american-time-use-survey,0,9,2025-08-02T13:50:25.846114,"[child care, elder care, time use, volunteering]",[],...,0,0,,,0,0,0,0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2509,Franchise Fund FY 2013 Annual Report,<p>Franchise Fund FY 2013 Annual Report</p>,Department of Veterans Affairs,VA-OM-OM-032,franchise-fund-fy-2013-annual-report,0,1,2025-08-08T08:20:27.163330,"[annual report, franchise fund, fy 2013]",[Financial],...,0,0,,,0,0,0,0,0.0,0.000000
2510,NNDSS - Table II. Chlamydia to Coccidioidomycosis,NNDSS - Table II. Chlamydia to Coccidioidomyco...,Centers for Disease Control and Prevention,https://data.cdc.gov/api/views/n835-hpyp,nndss-table-ii-chlamydia-to-coccidioidomycosis...,0,0,2025-08-08T22:07:07.949127,"[2016, chlamydia trachomatis infection, coccid...",[NNDSS],...,0,0,,,0,0,0,0,0.0,0.000000
2511,U.S. State and Territorial Public Mask Mandate...,"State and territorial executive orders, admini...",Centers for Disease Control and Prevention,https://data.cdc.gov/api/views/42jj-z7fa,u-s-state-and-territorial-public-mask-mandates...,0,0,2025-08-08T19:47:44.252563,"[covid-19, executive order, government order, ...",[Policy Surveillance],...,0,0,accessible; community; covid-19; excluded; tribal,,4,5,1,1,0.8,0.011396
2512,Submerged Land Act (SLA) Boundary,Homeland Infrastructure Foundation-Level Data ...,HIFLD,MGMT-GMO-HIFLD-407158,submerged-land-act-sla-boundary,0,0,2025-08-02T09:02:39.857358,[none],[],...,0,0,,,0,0,0,0,0.0,0.000000


# Next: 06_modeling